# DSA 504 — Class 9
## Visualization II: seaborn (Lab)

**Date:** Wednesday, Sep 30
**Today:** HW2 is due before class.

---

Class 8 covered matplotlib — the foundational plotting library. Today we cover **seaborn**, which is built on top of matplotlib but adds two things: nicer default styling, and chart types built specifically for statistical/exploratory work with tables of data (exactly the shape of data pandas gives you).

This is a lab session — more hands-on time, less new lecture material than usual.

### Learning goals
By the end of this class, you will be able to:
- Explain what seaborn adds on top of matplotlib, and when to reach for each
- Create bar plots, box plots, histograms, and scatter plots using seaborn's simpler syntax
- Use the `hue` parameter to add a third (categorical) dimension to a chart
- Choose between a box plot and a histogram for comparing distributions
- Customize a seaborn plot using the matplotlib functions underneath it


In [ ]:
#Install seaborn if you haven't already
%pip install seaborn

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sales = pd.read_csv("retail_sales_clean.csv")
sales["date"] = pd.to_datetime(sales["date"])
sales.head()


## 1. What seaborn Adds on Top of matplotlib

Recall from Class 8: a matplotlib bar chart required you to first compute the grouped values yourself (`.groupby().sum()`), then hand the result to `ax.bar()`. seaborn often does that grouping step *for you*, directly from a DataFrame — you just tell it which columns to use.


In [ ]:
sns.set_theme(style="whitegrid")   # a one-time call that improves the look of every chart below

# Compare: matplotlib required grouping first
revenue_by_store = sales.groupby("store")["revenue"].sum()
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(revenue_by_store.index, revenue_by_store.values)
ax.set_title("matplotlib: revenue by store (grouped manually)")
plt.show()

# seaborn: pass the raw DataFrame and column names directly -- it aggregates automatically
fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(data=sales, x="store", y="revenue", estimator="sum", errorbar=None, ax=ax)
ax.set_title("seaborn: revenue by store (aggregated automatically)")
plt.show()


**Note the `estimator="sum"` and `errorbar=None` arguments above.** By default, `sns.barplot()` shows the *mean* of `y` per group, plus a confidence interval (the small vertical line on each bar) — useful for statistical work, but not what we want when comparing totals. Setting `estimator="sum"` changes what's being aggregated, and `errorbar=None` turns off the confidence interval line. This default behavior is the single biggest difference from matplotlib's bar chart, and it's worth understanding rather than fighting.


In [ ]:
# The DEFAULT behavior: average revenue per store, with a confidence interval shown
fig, ax = plt.subplots(figsize=(6, 4))
sns.barplot(data=sales, x="store", y="revenue", ax=ax)
ax.set_title("Average Revenue per Transaction, by Store (with 95% CI)")
plt.show()


## 2. Box Plots — Comparing Distributions Across Groups

This is a genuinely new chart type, not available directly in matplotlib without manual work. A box plot shows the spread of a numeric variable *for each category at once* — the median, the middle 50% of values (the box), and outliers — all in one compact view.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=sales, x="category", y="revenue", ax=ax)
ax.set_title("Revenue Distribution by Category")
ax.tick_params(axis="x", rotation=20)
plt.show()


**Reading a box plot:** the line inside the box is the median. The box itself spans the middle 50% of the data (25th to 75th percentile). The "whiskers" extend to the typical range, and individual dots beyond them are flagged as outliers. This single chart answers "which category has the most variable revenue?" and "which category has outlier transactions?" — both questions a bar chart (which only shows one number per group) can't answer at all.


## 3. Distributions: `histplot()` and `kdeplot()`

seaborn's histogram function looks similar to matplotlib's, but adds one useful option: overlaying a smooth density curve.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(data=sales, x="units_sold", bins=30, kde=True, ax=ax)
ax.set_title("Distribution of Units Sold (with density curve)")
plt.show()


The smooth curve (the **KDE**, or kernel density estimate) is a smoothed-out version of the histogram — useful for seeing the overall shape without the bin-count sensitivity we discussed in Class 8.


In [ ]:
# Comparing distributions across categories on ONE chart using hue
fig, ax = plt.subplots(figsize=(8, 5))
sns.histplot(data=sales, x="revenue", hue="category", kde=True, alpha=0.4, ax=ax)
ax.set_title("Revenue Distribution by Category")
plt.show()


**This is the `hue` parameter** — it colors the chart by a categorical column, letting you compare groups directly on one plot instead of making five separate charts. You'll use `hue` constantly in seaborn; it's arguably the single most useful feature the library adds.


## 4. Scatter and Line Plots, with `hue`


In [ ]:
# Scatter plot colored by category
fig, ax = plt.subplots(figsize=(8, 5))
sns.scatterplot(data=sales, x="units_sold", y="revenue", hue="category", alpha=0.5, ax=ax)
ax.set_title("Units Sold vs. Revenue, by Category")
plt.show()


In [ ]:
# Line plot -- seaborn automatically aggregates and shows a confidence band if there
# are multiple values per x position (unlike matplotlib, which just connects raw points)
daily = sales.groupby(["date", "store"])["revenue"].sum().reset_index()

fig, ax = plt.subplots(figsize=(11, 5))
sns.lineplot(data=daily, x="date", y="revenue", hue="store", ax=ax)
ax.set_title("Daily Revenue by Store")
plt.show()


## 5. Violin Plots — A Box Plot with More Shape Detail

A violin plot combines the idea of a box plot with a KDE — showing the full shape of the distribution, mirrored on both sides, instead of just a box and whiskers.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.violinplot(data=sales, x="category", y="units_sold", ax=ax)
ax.set_title("Units Sold Distribution by Category")
ax.tick_params(axis="x", rotation=20)
plt.show()


**Box plot vs. violin plot — when to use which:** a box plot is more compact and easier to read at a glance across many categories; a violin plot shows more nuance (e.g., a distribution with two separate humps, which a box plot would hide entirely). For a quick comparison, default to box plots; reach for violin plots when you suspect the shape itself is interesting, not just the median and spread.


## 6. Customizing seaborn Plots with matplotlib

Every seaborn function returns a matplotlib `Axes` object — the exact same object type from Class 8. This means everything you already know (`.set_title()`, `.set_xlabel()`, `.legend()`) still works directly on top of a seaborn chart.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
sns.boxplot(data=sales, x="category", y="revenue", ax=ax)

# Everything below is plain matplotlib, applied to seaborn's output
ax.set_title("Revenue Distribution by Category", fontsize=14, fontweight="bold")
ax.set_xlabel("Product Category")
ax.set_ylabel("Revenue ($)")
ax.tick_params(axis="x", rotation=20)
plt.tight_layout()
plt.show()


---
## Guided Practice

Work through these using `sales`, already loaded above from `retail_sales_clean.csv`.


### Exercise 1 — Bar plot with sum
Create a seaborn bar plot showing total (not average) `units_sold` per store. Remember the `estimator` and `errorbar` arguments from Section 1.


In [ ]:
# Exercise 1 — your code here



### Exercise 2 — Box plot
Create a box plot comparing `units_sold` (not revenue) across the 4 stores. Which store appears to have the widest spread?


In [ ]:
# Exercise 2 — your code here



### Exercise 3 — Histogram with hue
Create a histogram of `revenue`, colored by `store` using `hue`. Set `alpha` low enough that overlapping distributions are still visible.


In [ ]:
# Exercise 3 — your code here



### Exercise 4 — Scatter with hue
Create a scatter plot of `units_sold` vs. `revenue`, colored by `store` instead of `category` this time.


In [ ]:
# Exercise 4 — your code here



### Exercise 5 (stretch) — Combine and customize
Create a violin plot of `revenue` by `category`, then use matplotlib methods to: set a bold title, rotate the x-axis labels 30 degrees, and label the y-axis as "Revenue ($)".


In [ ]:
# Exercise 5 — your code here



---
## Wrap-up

**Recap:** seaborn builds on matplotlib and adds automatic aggregation from a DataFrame, new chart types (box plots, violin plots), and the `hue` parameter for adding a categorical dimension to almost any chart. Every seaborn chart still returns a matplotlib `Axes`, so all of Class 8's customization tools still apply directly.

**Common mistakes to watch for:**
- Forgetting that `sns.barplot()` defaults to showing the *mean* with a confidence interval, not a sum — a frequent source of "why does my bar chart look different from my `.groupby().sum()` result?" confusion
- Overusing `hue` on a chart with too many categories, making it unreadable — a handful of groups works well; a dozen usually doesn't
- Reaching for a violin plot by default when a simpler box plot would communicate the same thing more clearly

**HW2 is due before this class.**

**Before next class (Oct 5):** Class 10 moves into programmatic data access — reading and writing different file formats (CSV, JSON, Excel, Parquet) beyond just the CSV we've used so far.
